# Exploratory data analysis 3

**INPUT**: Tokenized train, validation, test corpora

**OUTPUT**: Model parameter recommendations

| Step | Decision | Status | Comment |
|------|----------|--------|---------|
| Tokenized sequence length distribution | - | Pending | - |
| Vocabulary coverage analysis | - | Pending | - |
| OOV rate | - | - | - |
| Token statistics | - | Pending | - |

## Input & Setup

### Imports

In [ ]:
import pickle
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
import sys
from pathlib import Path

project_root = Path().resolve().parent
src = project_root / "src"
if str(src) not in sys.path:
    sys.path.insert(0, str(src))

from config.config import PATHS

### Load Pickles Into Dictionary

In [ ]:
paths = {
    "train": PATHS.train_pieces,
    "val": PATHS.validation_pieces,
    "test": PATHS.test_pieces
}

tokenized_data = {}
for split, path in paths.items():
    with open(path, "rb") as f:
        tokenized_data[split] = pickle.load(f)
    print(f"{split} split: {len(tokenized_data[split])} sequences")

## Steps

### Tokenized Sequence Length Distribution

In [ ]:
plt.figure(figsize=(12,6))

bins = 50
alpha = 0.5
for split, data in tokenized_data.items():
    lengths = [len(seq) for seq in data]
    plt.hist(lengths, bins=bins, alpha=alpha, label=split, edgecolor='black')

plt.title("Tokenized Sequence Length Distribution by Split")
plt.xlabel("Sequence Length (tokens)")
plt.ylabel("Count")
plt.legend()
plt.grid(axis='y', alpha=0.75)
plt.show()

for split, data in tokenized_data.items():
    lengths = [len(seq) for seq in data]
    print(f"{split} — mean: {pd.Series(lengths).mean():.2f}, median: {pd.Series(lengths).median()}, min: {min(lengths)}, max: {max(lengths)}")

### Vocabulary Coverage

In [ ]:
# Flatten training tokens
train_tokens = [token for seq in tokenized_data["train"] for token in seq]

# Count token frequencies
train_token_counts = Counter(train_tokens)
total_train_tokens = len(train_tokens)

# Create DataFrame for coverage
vocab_df = pd.DataFrame(train_token_counts.items(), columns=["token", "count"]).sort_values(by="count", ascending=False)
vocab_df["cum_freq"] = vocab_df["count"].cumsum() / total_train_tokens

# Plot cumulative coverage
plt.figure(figsize=(10,6))
plt.plot(range(1, len(vocab_df)+1), vocab_df["cum_freq"])
plt.title("Vocabulary Coverage (Training Split)")
plt.xlabel("Token Rank")
plt.ylabel("Cumulative Coverage")
plt.grid(True)
plt.show()

print("Top 30 tokens in training set:")
print(vocab_df.head(30))

### OOV Rate